# 32. LightRAG

Dual-level retrieval: match specific entities (low level) and broad themes (high level), both through the graph.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

LightRAG simplifies GraphRAG with **dual-level retrieval**. *Low-level* keywords (specific entities) are matched against an **entity** index and expanded through their graph neighbours; *high-level* keywords (abstract themes) are matched against a **relation** index and mapped back to source text. Merging both gives specific facts and broad context without GraphRAG's heavy community reports.

## How it works

1. Extract triples and build the graph
2. Index entities (nodes) and relations (edges) separately
3. LLM splits the query into low-level and high-level keywords
4. Low-level keywords -> entity index -> graph neighbours
5. High-level keywords -> relation index -> source chunks
6. Merge and generate

```text
 query --> LLM --> low-level: [specific entities]  --> entity index  --> graph neighbours \
                   high-level: [themes/concepts]   --> relation index --> relations+chunks  --> merge --> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install networkx`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.Graph()
relation_text = []
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r, chunk=i)
        relation_text.append((f"{s} {r} {o}", i))

entity_store = build_store(list(G.nodes))                              # low-level index
relation_store = build_store([t for t, _ in relation_text], [{"chunk": c} for _, c in relation_text])  # high-level index
print(len(entity_store), "entities,", len(relation_store), "relations")

An LLM splits the query into **low-level keywords** (entities, details) and **high-level keywords** (themes, concepts).

In [ ]:
def keywords(query):
    out = llm_json(f'Extract keywords. Reply as JSON {{"low": [specific entities], "high": [abstract themes]}}.\n\nQuestion: {query}')
    return (out.get("low", []), out.get("high", [])) if isinstance(out, dict) else ([query], [query])

def light_rag(query, k=3):
    low, high = keywords(query)
    context = set()
    for kw in low:                                    # entity -> its relations
        for h in entity_store.search(kw, k):
            node = h["text"]
            for nb in G.neighbors(node):
                context.add(f"{node} {G[node][nb]['relation']} {nb}")
    for kw in high:                                   # theme -> matching relations -> source chunks
        for h in relation_store.search(kw, k):
            context.add(h["text"])
            context.add(chunks[h["meta"]["chunk"]])
    return low, high, answer(query, sorted(context))

low, high, response = light_rag("How do hybrid search and reranking work together to improve retrieval?")
print("low :", low); print("high:", high); print(response)

## Inspect the keyword split
This is where LightRAG differs from plain GraphRAG.

In [ ]:
for q in ["How do hybrid search and reranking work together?", "What is HNSW?", "Why do teams combine fine-tuning with retrieval?"]:
    low, high = keywords(q)
    print(f"{q}\n   low : {low}\n   high: {high}\n")

## What does each level contribute?
Run the retrieval with only one level at a time.

In [ ]:
def levels_only(query, which, k=3):
    low, high = keywords(query)
    ctx = set()
    if which == "low":
        for kw in low:
            for h in entity_store.search(kw, k):
                for nb in G.neighbors(h["text"]): ctx.add(f"{h['text']} {G[h['text']][nb]['relation']} {nb}")
    else:
        for kw in high:
            for h in relation_store.search(kw, k): ctx.add(h["text"]); ctx.add(chunks[h["meta"]["chunk"]])
    return sorted(ctx)

q = "How do hybrid search and reranking work together?"
for which in ["low", "high"]:
    c = levels_only(q, which)
    print(f"{which}-level context: {len(c)} items, {sum(map(len, c))} chars")

## Compare against vector RAG

In [ ]:
compare_pipelines({"vector RAG": lambda q: rag(q, store, 3)[0], "LightRAG": lambda q: light_rag(q)[2]}, n=4)

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Cheaper than community-report GraphRAG | Depends on keyword-extraction quality |
| Handles both specific and abstract queries | Still needs LLM triple extraction |
| Incremental updates (add triples without rebuilding reports) | Merged context can be large |

## When to use it

Mixed question types over a graph-structured corpus where GraphRAG's global reports are too costly.

## Experiments to try

- Weight low vs high level results differently.
- Cache keyword extraction.
- Add a re-ranker on the merged context.

## Key takeaways

- Two indexes, two levels of query intent.
- Cheaper than full GraphRAG.
- Keyword splitting is the key step.

---
**Next:** [33. GraphRAG with Local Attribution](../33_graph_rag_local_attribution/graph_rag_local_attribution.ipynb)